In [1]:
# Jupyter Notebook: ETL for GameFact (Tior Games)
# Required packages: pandas, openpyxl (for Excel), pyodbc or sqlalchemy (for DB), etc.

import pandas as pd
import pyodbc

In [3]:
# 1. Database Connection Setup

conn_str = (
    "Driver={ODBC Driver 18 for SQL Server};"
    "SERVER=35.246.21.86;"
    "Database=TiorGames;"
    "UID=UoP167;"
    "PWD=DWHu3p0253;"
    "TrustServerCertificate=yes;"
)
conn = pyodbc.connect(conn_str)

In [4]:
# 2. Load Dimension Data (for foreign key checks)

dim_date   = pd.read_sql("SELECT DateID, DateYear FROM DateDim;", conn)
dim_event  = pd.read_sql("SELECT EventID FROM EventDim;", conn)
dim_stadium= pd.read_sql("SELECT StadiumID FROM StadiumDim;", conn)
dim_referee= pd.read_sql("SELECT RefereeID FROM RefereeDim;", conn)
dim_pause  = pd.read_sql("SELECT PauseID FROM PauseDim;", conn)
dim_highlight = pd.read_sql("SELECT HighlightID FROM HighlightDim;", conn)

C:\Users\Mulen\AppData\Local\Temp\ipykernel_2212\1019081788.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dim_date   = pd.read_sql("SELECT DateID, DateYear FROM DateDim;", conn)
C:\Users\Mulen\AppData\Local\Temp\ipykernel_2212\1019081788.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dim_event  = pd.read_sql("SELECT EventID FROM EventDim;", conn)
C:\Users\Mulen\AppData\Local\Temp\ipykernel_2212\1019081788.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dim_stadium= pd.read_sql("SELECT StadiumID FROM StadiumDi

In [15]:
# If you want them as sets for faster membership checks:
set_dateID      = set(dim_date['DateID'])
date_2024       = set(dim_date.loc[dim_date['DateYear']==2024, 'DateID'])
set_eventID     = set(dim_event['EventID'])
set_stadiumID   = set(dim_stadium['StadiumID'])
set_refereeID   = set(dim_referee['RefereeID'])
set_pauseID     = set(dim_pause['PauseID'])
set_highlightID = set(dim_highlight['HighlightID'])

In [5]:
# 3. Extract Data GameFact_MockData.xlsx

excel_path = 'C:\\Users\\Mulen\\Downloads\\GameFact_MockData.xlsx'  # Adjust path as needed
df_excel = pd.read_excel(excel_path)   

# Expect columns: [EventID, StadiumID, RefereeID, GameID, TimeID, DateID, PauseID, HighlightID, 
#                  GameDuration, GameNumberOfPause, GameInterruption, GameMinuteOfPause, 
#                  GameDurationOfPause, GameResult]

print("Preview of Excel Data:")
display(df_excel.head())

Preview of Excel Data:


,EventID,StadiumID,RefereeID,GameID,TimeID,DateID,PauseID,HighlightID,GameDuration,GameNumberOfPause,GameTeam,GameInterruption,GameMinuteOfPause,GameDurationOfPause,GameResult
0,10,10,2.0,86,5050,157,NaN,2.0,55,NaN,blue,1,NaN,NaN,red wins
1,10,10,5.0,89,4674,,2.0,10.0,34,3.0,,1,24.0,23.0,blue wins
2,14,10,14.0,88,,,NaN,NaN,19,NaN,,0,NaN,NaN,
3,11,10,1.0,,6000,,NaN,3.0,-5,2.0,red,-1,NaN,NaN,
4,10,11,3.0,91,7000,,4.0,5.0,6600,4.0,green,1,30.0,15.0,yellow


In [11]:
# 3. Extract Data GameFactMockData2.json

import json
import re

json_path = 'C:\\Users\\Mulen\\Downloads\\GameFactMockData2.json'

with open(json_path, "r", encoding="utf-8") as f:
    data_str = f.read()

# Replace the invalid snippet `"DateID": ,` with `"DateID": null,`
# (You might do more sophisticated patterns if there are multiple lines with the same issue.)
data_str = data_str.replace('"DateID": ,', '"DateID": null,')

# Now parse
raw_json = json.loads(data_str)

# The JSON structure: { "gamefact_data": [ {...}, {...}, ... ] }
df_json = pd.json_normalize(raw_json["gamefact_data"])

print("\nPreview of JSON Data:")
display(df_json.head())



Preview of JSON Data:


,EventID,StadiumID,RefereeID,GameID,TimeID,DateID,PauseID,HighlightID,GameDuration,GameNumberOfPause,GameTeam,GameInterruption,GameMinuteOfPause,GameDurationOfPause,GameResult
0,1,10,2,86,5050,Invalid,None,2,55,None,blue,1,None,None,red wins
1,10,10,5,89,4674,1500,2,10,34,3,blue,1,24,23,blue wins
2,10,10,14,88,15426,Invalid,None,None,19,None,blue,0,None,None,red wins
3,10,10,1,90,6000,15,None,3,-5,2,red,-1,None,None,blue wins
4,100,11,3,91,7000,Invalid,4,5,45,4,green,1,30,15,yellow


In [ ]:
# Let's define a function to clean & validate a DataFrame of "GameFact-like" rows.

def validate_gamefact_df(df_source, source_name="Unknown"):
    """
    Applies the 5 rules:
    1. No null or "None" fields for required columns.
    2. Columns that should be int must parse as int.
    3. DateID must exist in dim_date (and be year 2024).
    4. Other foreign keys must exist in respective dims.
    5. Optionally check positivity or range for GameDuration, etc.
    
    Returns: Two DataFrames: (valid_rows, invalid_rows) with a reason for invalid.
    """
    required_int_cols = [
        "EventID","StadiumID","RefereeID","GameID","TimeID","DateID",
        "PauseID","HighlightID","GameDuration","GameNumberOfPause",
        "GameInterruption","GameMinuteOfPause","GameDurationOfPause"
    ]
    
    # Ensure all those columns exist, fill missing with NaN to unify approach
    for c in required_int_cols:
        if c not in df_source.columns:
            df_source[c] = None  # if missing column entirely
    
    # Make a copy so we don't alter original
    df = df_source.copy()
    df["InvalidReason"] = ""
    
    # 1. Check for null or "None"
    for c in required_int_cols:
        # If any row has None or "None" or is just empty, it's invalid
        mask_null = df[c].isnull() | (df[c].astype(str).str.strip().isin(["None", ""]))
        df.loc[mask_null, "InvalidReason"] += f"[{c} is null/None]; "
    
    # 2. Convert to integer if possible, or mark invalid
    for c in required_int_cols:
        # We'll try a function that attempts to convert and if fails, mark invalid
        def try_int(x):
            try:
                return int(float(x))  # handle "10.0"
            except:
                return None
        df[f"{c}_int"] = df[c].apply(try_int)
        # Mark invalid if the result is None
        mask_invalid = df[f"{c}_int"].isnull()
        df.loc[mask_invalid, "InvalidReason"] += f"[{c} not an integer]; "
    
    # 3. Check DateID membership in date_2024
    # We'll do this after ensuring we have integer version
    df["DateID_valid"] = df["DateID_int"].apply(lambda x: x in date_2024 if x is not None else False)
    df.loc[~df["DateID_valid"], "InvalidReason"] += "[DateID not in 2024 or invalid]; "
    
    # 4. Check foreign keys in dimension sets
    # EventID
    df["EventID_valid"] = df["EventID_int"].apply(lambda x: x in set_eventID if x is not None else False)
    df.loc[~df["EventID_valid"], "InvalidReason"] += "[EventID not found]; "
    # StadiumID
    df["StadiumID_valid"] = df["StadiumID_int"].apply(lambda x: x in set_stadiumID if x is not None else False)
    df.loc[~df["StadiumID_valid"], "InvalidReason"] += "[StadiumID not found]; "
    # RefereeID
    df["RefereeID_valid"] = df["RefereeID_int"].apply(lambda x: x in set_refereeID if x is not None else False)
    df.loc[~df["RefereeID_valid"], "InvalidReason"] += "[RefereeID not found]; "
    # PauseID
    df["PauseID_valid"] = df["PauseID_int"].apply(lambda x: x in set_pauseID if x is not None else False)
    df.loc[~df["PauseID_valid"], "InvalidReason"] += "[PauseID not found]; "
    # HighlightID
    df["HighlightID_valid"] = df["HighlightID_int"].apply(lambda x: x in set_highlightID if x is not None else False)
    df.loc[~df["HighlightID_valid"], "InvalidReason"] += "[HighlightID not found]; "
    
    # 5. Optional: filter out negative or excessive GameDuration
    # e.g., let's say 1 <= GameDuration <= 300
    df["GameDuration_valid"] = df["GameDuration_int"].apply(lambda x: (x>=1 and x<=300) if x is not None else False)
    df.loc[~df["GameDuration_valid"], "InvalidReason"] += "[GameDuration out of plausible range]; "
    
    # Now define a valid row as one with no "InvalidReason"
    df["IsValid"] = df["InvalidReason"].apply(lambda reason: (reason.strip() == ""))
    
    # Create final cleaned columns for easy mapping:
    # We'll rename all the *_int columns back to original
    final_cols = [
        "EventID","StadiumID","RefereeID","GameID","TimeID","DateID",
        "PauseID","HighlightID","GameDuration","GameNumberOfPause",
        "GameInterruption","GameMinuteOfPause","GameDurationOfPause","GameResult"
    ]
    
    # For numeric columns, we override with the integer version
    for c in required_int_cols:
        df.loc[df["IsValid"], c] = df.loc[df["IsValid"], f"{c}_int"]
    
    # We'll keep "GameResult" as is, just ensure null or None is flagged
    df["GameResult"] = df["GameResult"].fillna("").astype(str)
    mask_result_null = df["GameResult"].isin(["None",""])
    df.loc[mask_result_null, "InvalidReason"] += "[GameResult missing]; "
    df.loc[mask_result_null, "IsValid"] = False
    
    # Re-check for final validity after that
    df["IsValid"] = df["InvalidReason"].apply(lambda reason: (reason.strip() == ""))
    
    # Return separate dataframes
    valid_df = df[df["IsValid"]].copy()
    invalid_df = df[~df["IsValid"]].copy()
    
    # Provide some context
    valid_df["Source"] = source_name
    invalid_df["Source"] = source_name
    
    return (valid_df, invalid_df)

In [ ]:
# Validate the Excel dataset

valid_excel, invalid_excel = validate_gamefact_df(df_excel, source_name="Excel")

print("\nValid rows from Excel:", len(valid_excel))
print("Invalid rows from Excel:", len(invalid_excel))


Valid rows from Excel: 0
Invalid rows from Excel: 20


In [ ]:
# Validate the JSON dataset

valid_json, invalid_json = validate_gamefact_df(df_json, source_name="JSON")

print("\nValid rows from JSON:", len(valid_json))
print("Invalid rows from JSON:", len(invalid_json))


Valid rows from JSON: 1
Invalid rows from JSON: 14


In [ ]:
# Combine valid rows into one DataFrame

valid_combined = pd.concat([valid_excel, valid_json], ignore_index=True)
invalid_combined = pd.concat([invalid_excel, invalid_json], ignore_index=True)

print("\nTotal valid rows combined:", len(valid_combined))
print("Total invalid rows combined:", len(invalid_combined))


Total valid rows combined: 1
Total invalid rows combined: 34


In [ ]:
# 7. Generate INSERT statements for valid rows

# We'll define a function to produce a list of SQL inserts.
def build_insert_statements(df: pd.DataFrame, table_name="GameFact") -> list:
    """
    Generate INSERT INTO table_name (...) VALUES (...) statements
    for each row in df.
    """
    statements = []
    for idx, row in df.iterrows():
        # Build the column list
        col_list = [
            "EventID","StadiumID","RefereeID","GameID","TimeID","DateID","PauseID","HighlightID",
            "GameDuration","GameNumberOfPause","GameInterruption","GameMinuteOfPause",
            "GameDurationOfPause","GameResult"
        ]
        
        # Create a tuple of values, ensuring strings are properly quoted
        values = []
        for col in col_list:
            val = row[col]
            if isinstance(val, str):
                val_escaped = val.replace("'", "''")  # escape single quotes
                values.append(f"'{val_escaped}'")
            else:
                values.append(str(val) if pd.notnull(val) else "NULL")
        
        col_str = ", ".join(col_list)
        val_str = ", ".join(values)
        
        stmt = f"INSERT INTO {table_name} ({col_str}) VALUES ({val_str});"
        statements.append(stmt)
    return statements

insert_stmts = build_insert_statements(valid_combined, "GameFact")

print("\nSample INSERT statement:")
if len(insert_stmts) > 0:
    print(insert_stmts[0])


Sample INSERT statement:
INSERT INTO GameFact (EventID, StadiumID, RefereeID, GameID, TimeID, DateID, PauseID, HighlightID, GameDuration, GameNumberOfPause, GameInterruption, GameMinuteOfPause, GameDurationOfPause, GameResult) VALUES (10.0, 10, 5.0, 89, 4674.0, 1500.0, 2.0, 10.0, 34, 3.0, 1.0, 24.0, 23.0, 'blue wins');


In [ ]:
# 9. Final Logging of Rejected Rows

# Save invalid rows to a CSV or table for review
invalid_combined[[
    "Source","EventID","StadiumID","RefereeID","GameID","TimeID","DateID",
    "PauseID","HighlightID","GameDuration","GameNumberOfPause","GameInterruption",
    "GameMinuteOfPause","GameDurationOfPause","GameResult","InvalidReason"
]].to_csv("GameFact_LoadErrors.csv", index=False)

print("\nRejected rows saved to 'GameFact_LoadErrors.csv'. Done!")